# Módulo 1: Paisajes de Aptitud y la Dificultad del Muestreo a Ciegas
## Tutorial de Algoritmos Genéticos para Investigación Doctoral

### 1. Motivación Teórica
En la investigación científica e ingenieril moderna, muchos problemas de optimización son **problemas de caja negra (Black-Box Optimization)**: no poseemos una expresión matemática analítica diferenciable de la función objetivo ni conocemos sus derivadas. Solo podemos suministrar un vector de entrada $\mathbf{x} \in \mathbb{R}^D$ y recibir una medición de rendimiento escalar $f(\mathbf{x})$ (por ejemplo, el resultado de una simulación de elementos finitos, el error de validación de un modelo, o una medición en laboratorio).

El concepto de **Paisaje de Aptitud (*Fitness Landscape*)**, introducido por Sewall Wright en 1932, modela el espacio de búsqueda como un terreno topológico multidimensional donde la altura representa la calidad de cada solución.

Cuando el paisaje es **altamente multimodal y deceptivo**, los métodos clásicos (gradiente descendente, ascenso de colinas, o búsqueda puramente aleatoria) colapsan estrepitosamente.

### 2. La Función F6 de Schaffer con Frecuencia Calibrable ($\omega$)
Introducida por J. David Schaffer (1989), es un referente obligado en computación evolutiva para evaluar la capacidad de exploración global vs. explotación local:

$$
f(x, y) = 0.5 - \frac{\sin^2\left(\omega \cdot \sqrt{(x-x_0)^2 + (y-y_0)^2}\right) - 0.5}{\left[1 + 0.001\cdot\left((x-x_0)^2 + (y-y_0)^2\right)\right]^2}
$$

Donde:
- $(x_0, y_0)$ es la ubicación del **óptimo global**.
- $\omega$ es la **frecuencia de oscilación** (calibrable en la interfaz). A mayor $\omega$, las ondas son más densas y el pico central se comprime con radio $r_{pico} \approx \frac{\pi}{2\omega}$.
- El valor máximo teórico alcanzable es $f(x_0, y_0) = 1.0$ independientemente de $\omega$.

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Añadir la carpeta python al path
sys.path.append(os.path.abspath("../python"))
from benchmarks import SchafferF6, OptimizationProblem

print("Módulos importados correctamente.")

### 3. Impacto de la Frecuencia $\omega$ en la Topología del Terreno
Grafiquemos tres paisajes con $\omega = 0.5$ (ondas suaves), $\omega = 1.0$ (estándar), y $\omega = 3.0$ (alta deceptividad):

In [ ]:
omegas = [0.5, 1.0, 3.0]
titles = ["Ondas Suaves (ω=0.5)", "Estándar F6 (ω=1.0)", "Alta Deceptividad (ω=3.0)"]

plt.figure(figsize=(15, 4.5), dpi=130)

for idx, (om, tit) in enumerate(zip(omegas, titles), 1):
    prob = OptimizationProblem("schaffer_f6", center_mode="fixed", fixed_x0=0.0, fixed_y0=0.0, frequency=om)
    X, Y, Z = prob.evaluate_mesh(num_points=180)
    
    plt.subplot(1, 3, idx)
    c = plt.contourf(X, Y, Z, levels=50, cmap="viridis")
    plt.plot(0, 0, 'r*', markersize=12, label="Óptimo (0,0)")
    plt.title(tit, fontsize=11, fontweight="bold")
    plt.xlabel("X")
    plt.ylabel("Y")
    if idx == 3:
        plt.colorbar(c, label="Aptitud (Fitness)")

plt.tight_layout()
plt.show()

### 4. Zoom en el Pico Central y la Aguja en el Pajar
Observa cómo a $\omega = 3.0$, la cuenca de atracción del óptimo global es extremadamente reducida comparada con $\omega = 1.0$:

In [ ]:
plt.figure(figsize=(12, 5), dpi=120)

for idx, om in enumerate([1.0, 3.0], 1):
    prob = OptimizationProblem("schaffer_f6", center_mode="fixed", fixed_x0=0.0, fixed_y0=0.0, frequency=om)
    xs = np.linspace(-10, 10, 150)
    ys = np.linspace(-10, 10, 150)
    X, Y = np.meshgrid(xs, ys)
    Z = SchafferF6.evaluate_grid(X, Y, 0.0, 0.0, frequency=om)
    
    plt.subplot(1, 2, idx)
    cf = plt.contourf(X, Y, Z, levels=40, cmap="plasma")
    plt.colorbar(cf, label="Aptitud")
    plt.plot(0, 0, 'w*', markersize=14, label="Pico Óptimo")
    r_basin = np.pi / (2 * om)
    circ = plt.Circle((0, 0), r_basin, color='white', fill=False, linestyle='--', linewidth=1.5, label=f"Cuenca (r={r_basin:.2f})")
    plt.gca().add_patch(circ)
    plt.title(f"Zoom [-10, 10] con ω = {om}", fontsize=11)
    plt.legend(loc="upper right", fontsize=9)

plt.tight_layout()
plt.show()

### 5. Experimento: Muestreo a Ciegas (Fuerza Bruta / Monte Carlo)
Imagina que eres un algoritmo sin inteligencia evolutiva. Tu única estrategia es generar puntos aleatorios uniformes en el dominio $[-100, 100] \times [-100, 100]$.

Probemos con diferentes presupuestos de evaluación $N \in \{50, 200, 1000\}$ para $\omega = 2.5$:

In [ ]:
budgets = [50, 200, 1000]
np.random.seed(42)

print(f"{'Presupuesto (N)':<18} | {'Mejor Aptitud':<15} | {'% del Óptimo':<15} | {'Distancia al Óptimo':<20}")
print("-" * 75)

for b in budgets:
    prob_exp = OptimizationProblem("schaffer_f6", center_mode="fixed", fixed_x0=0.0, fixed_y0=0.0, frequency=2.5)
    best_fit = -1.0
    best_pt = None
    
    for _ in range(b):
        rx, ry = prob_exp.generate_random_individual()
        res = prob_exp.evaluate(rx, ry)
        if res["fitness"] > best_fit:
            best_fit = res["fitness"]
            best_pt = res
            
    print(f"{b:<18} | {best_fit:<15.4f} | {best_pt['percent_of_global']:<15.2f}% | {best_pt['distance_to_optimum']:<20.2f}")

### 6. Preguntas de Investigación para el Estudiante de Doctorado

1. **Efecto de la Frecuencia $\omega$ en la Probabilidad de Éxito:**
   - Si el radio de la cuenca central es $r = \frac{\pi}{2\omega}$, demuestra analíticamente que el área de la cuenca decrece con $\mathcal{O}(\omega^{-2})$.
   - ¿Cómo afecta duplicar la frecuencia $\omega$ (ej. de 1.0 a 2.0) al número de evaluaciones aleatorias $N$ necesarias para mantener una probabilidad de éxito del $95\%$?

2. **Zoom y Exploración en la Interfaz Web:**
   - Abre `index.html` en tu navegador con $\omega = 2.5$.
   - Usa la herramienta de zoom (`🔍+` o rueda del ratón) para explorar las crestas concéntricas alrededor del centro.
   - Prueba ejecutar el Algoritmo Genético y observa cómo la población se sitúa en los anillos interiores.